In [ ]:
""" Lab 2 - Streaming.
Goal: Feel the diffrence between time-to-first-token (TTFT) and total time, and learn where errors can appear when streaming

"""

In [2]:
import time
import anthropic
from common import MODEL, banner, get_client

client = get_client()
PROMPT = "Write 250 word summary why entraprise adopt AI Agents"


In [ ]:
#######Non Streaming : User Sees nothing until it is done

t0 = time.perf_counter()

r = client.messages.create(model=MODEL, max_tokens=400,
                           messages = [{"role":"user","content":PROMPT}])

total_ns = time.perf_counter() - t0

print(f"First Visible text after {total:.2f}s (==total)")




In [31]:
print(f"{r.usage.input_tokens}in / {r.usage.output_tokens}")

29in / 400


In [7]:
time.perf_counter()

47387.7985071

In [ ]:
#####Streaming : First Token arrives early, total time is about the same

banner("Step#3 - Streaming")

t0 = time.perf_counter()
ttft = None
partial = []

try:
    with client.messages.stream(model=MODEL,max_tokens=600,messages=[{"role":"user","content": PROMPT}]) as stream:
        for chunk in stream.text_stream:
            if ttft is None:
                ttft = time.perf_counter() - t0
            partial.append(chunk)
            print(chunk,end="",flush=True)
        final = stream.get_final_message()
except anthropic.APIStatusError as exc:
    print(f"\n[stream error {exc.status_code}] partial chars kept: {len(''.join(partial))}")
    raise

except anthropic.APIConnectionError:
    print(f"\n[connection dropped] partial chars kept:{len(''.join(partial))}")
    raise
total_s = time.perf_counter() - t0

print(f"\n\nTTFT:{ttft:.2f}s | total:{total_s:.2f}s | non-stream first-text:{total_ns:.2f}s")
print(f"usage:", final.usage.input_tokens,"in /", final.usage.output_tokens,"out","| stop_reason",final.stop_reason)



======== Step#3 - Streaming ========
# Why Enterprises Are Adopting AI Agents

Enterprises are adopting AI agents because they offer a step beyond traditional automation and basic chatbots. Unlike rule-based tools, AI agents can reason through tasks, make decisions, use software tools, and complete multi-step workflows with minimal human oversight.

**Productivity and efficiency.** Agents take over repetitive, time-consuming work such as data entry, report generation, ticket triage, invoice processing, and scheduling. This frees employees to focus on strategic, creative, and relationship-driven work, while shortening turnaround times across departments.

**Cost reduction.** By automating complex processes, companies can lower operational costs, reduce errors, and scale output without proportionally growing headcount. Agents work around the clock, which is particularly valuable for customer support and IT operations.

**Better decision-making.** Agents can analyze large volumes of stru

In [ ]:
print("""
Ansawer in your own words:
1. Did Streaming make the answer faster or just feel faster? what did it change?
 Answer: First response is faster and overall not much change
2. Is Streaming Cheaper? (Check usage of Step2 and 3)
 Answer: No. Compare to non-stream it is high
3. why is "HTTP 200 returned" not proof the will complete?
  Answer: That will consume more token. 
)